In [ ]:
!nvidia-smi

In [ ]:
import os
# Set multiple environment variables to fix GPU issues
os.environ['TF_XLA_FLAGS'] = '--tf_xla_enable_xla_devices=false'
os.environ['XLA_FLAGS'] = '--xla_gpu_cuda_data_dir=/opt/conda'
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'

import tensorflow as tf
# Configure GPU memory growth
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

import tensorflow as tf
import cv2
import numpy as np
import matplotlib.pyplot as plt
import random
from keras_unet_collection import models
from tensorflow.keras.layers import *
from tensorflow.keras.models import Model

In [ ]:
image_dir = "../dataset/image"
mask_dir  = "../dataset/mask"

images = os.listdir(image_dir)
masks= os.listdir(mask_dir)

print("Total images:", len(images))
print("Total Masks",len(masks))

In [ ]:
sample = images[10]

img = cv2.imread(os.path.join(image_dir, sample))
print("Image shape:", img.shape)

plt.imshow(img)

In [ ]:
plt.figure(figsize=(15,10))

for i in range(6):

    sample = random.choice(images)

    prefix, rest = sample.split("_",1)
    mask_name = f"{prefix}_road_{rest}"

    img = cv2.imread(os.path.join(image_dir, sample))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    mask = cv2.imread(os.path.join(mask_dir, mask_name))
    mask = cv2.cvtColor(mask, cv2.COLOR_BGR2RGB)

    plt.subplot(3,4,i*2+1)
    plt.imshow(img)
    plt.title("Image")

    plt.subplot(3,4,i*2+2)
    plt.imshow(mask)
    plt.title("Mask")

plt.show()

In [ ]:
pixels = []

for img_name in images[:50]:
    img = cv2.imread(os.path.join(image_dir, img_name))
    pixels.append(img.mean())

print("Mean pixel intensity:", np.mean(pixels))

In [ ]:
plt.hist(pixels, bins=20)
plt.title("Image brightness distribution")
plt.show()

In [ ]:
for img_name in images[:5]:

    prefix, rest = img_name.split("_",1)
    mask_name = f"{prefix}_road_{rest}"

    print(img_name, "->", mask_name)

In [ ]:
heatmap = np.zeros((375,1242))

for img_name in images[:50]:

    prefix, rest = img_name.split("_",1)
    mask_name = f"{prefix}_road_{rest}"

    mask = cv2.imread(os.path.join(mask_dir, mask_name))

    road = mask[:,:,0] == 255

    road = cv2.resize(road.astype(np.uint8), (1242,375))

    heatmap += road

plt.imshow(heatmap, cmap="hot")
plt.title("Road pixel frequency")
plt.colorbar()
plt.show()

In [ ]:
def get_mask_name(img_name):
    prefix, rest = img_name.split("_",1)
    return f"{prefix}_road_{rest}"

In [ ]:
IMG_HEIGHT = 256
IMG_WIDTH = 832  

def load_sample(img_name):

    img_path = os.path.join(image_dir, img_name)
    mask_path = os.path.join(mask_dir, get_mask_name(img_name))

    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    mask = cv2.imread(mask_path)

    # road pixels
    road = mask[:,:,0] == 255
    mask = road.astype(np.float32)

    img = cv2.resize(img,(IMG_WIDTH, IMG_HEIGHT))
    mask = cv2.resize(mask, (IMG_WIDTH, IMG_HEIGHT), interpolation=cv2.INTER_NEAREST)

    mask = np.expand_dims(mask,axis=-1)

    return img, mask

In [ ]:
X=[]
Y=[]

for img_name in images:
    
    img,mask = load_sample(img_name)

    X.append(img)
    Y.append(mask)

X = np.array(X)/255.0
Y = np.array(Y).astype(np.float32)

print(X.shape,Y.shape)

In [ ]:
from sklearn.model_selection import train_test_split

X_train,X_val,Y_train,Y_val = train_test_split(
    X,Y,test_size=0.2,random_state=42
)

In [ ]:
model = models.unet_2d(
    (256, 832, 3),
    filter_num=[64,128,256,512],
    n_labels=1,
    activation='ReLU',
    output_activation='Sigmoid'
)

In [ ]:
def dice_loss(y_true, y_pred):
    smooth = 1e-6

    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    intersection = tf.reduce_sum(y_true * y_pred, axis=[1,2,3])
    union = tf.reduce_sum(y_true, axis=[1,2,3]) + tf.reduce_sum(y_pred, axis=[1,2,3])

    dice = (2. * intersection + smooth) / (union + smooth)
    return 1 - tf.reduce_mean(dice)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss=dice_loss,
    metrics=[
        tf.keras.metrics.BinaryAccuracy(),
        tf.keras.metrics.MeanIoU(num_classes=2)
    ]
)

model.summary()

In [ ]:
history = model.fit(
    X_train,
    Y_train,
    validation_data=(X_val,Y_val),
    epochs=15,
    batch_size=4
)

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(history.history['binary_accuracy'], label="Train Acc")
plt.plot(history.history['val_binary_accuracy'], label="Val Acc")
plt.legend()
plt.title("Accuracy")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(history.history['loss'], label="Train Loss")
plt.plot(history.history['val_loss'], label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()

plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(history.history['mean_io_u'], label="Train IoU")
plt.plot(history.history['val_mean_io_u'], label="Val IoU")
plt.legend()
plt.title("IoU")
plt.show()

In [ ]:
idx = 15
pred = model.predict(X_val[idx:idx+1])[0]
plt.figure(figsize=(12,4))
plt.subplot(1,3,1)
plt.imshow(X_val[idx])
plt.title("Image")
plt.subplot(1,3,2)
plt.imshow(Y_val[idx].squeeze())
plt.title("True Mask")
plt.subplot(1,3,3)
plt.imshow(pred.squeeze() > 0.5)
plt.title("Prediction")
plt.show()

In [ ]:
import tarfile

model.save("/home/sagemaker-user/code/Self-Driving_percerptron/models/model.keras")

with tarfile.open("../models/model.tar.gz", "w:gz") as tar:
    tar.add("/home/sagemaker-user/code/Self-Driving_percerptron/models/model.keras", arcname="model.keras")

In [ ]:
video_path = "/testing/challenge.mp4"

cap = cv2.VideoCapture(video_path)

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

print(width, height, fps)

In [ ]:
fourcc = cv2.VideoWriter_fourcc(*'mp4v')

out = cv2.VideoWriter("/kaggle/working/output.mp4",fourcc,fps,(width, height))

In [ ]:
IMG_SIZE = 256

def process_frame(frame):

    original = frame.copy()

    small = cv2.resize(frame, (IMG_SIZE, IMG_SIZE))
    small = small / 255.0

    pred = model.predict(small[np.newaxis,...])[0]

    mask = pred.squeeze() > 0.5

    mask = cv2.resize(mask.astype(np.uint8), (width, height))

    overlay = original.copy()
    overlay[mask == 1] = [0,255,0]

    return overlay

In [ ]:
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    result = process_frame(frame)
    out.write(result)
cap.release()
out.release()